# ASSISTments BKT calibration for EchoPilot

This notebook reproduces the learner-model training experiment used by the app. Gemini quiz generation is **not** trained here; this dataset is only used to fit the shared starting parameters for Bayesian Knowledge Tracing (BKT).

**Dataset:** corrected/collapsed ASSISTments 2009–2010 Skill Builder math responses. The corrected file has one row per student-problem; multi-skill labels are kept together. Its `correct` field is 1 for correct on the first attempt and 0 for incorrect on the first attempt or help requested. [Official dataset page](https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010).

The experiment splits by student, fits on at most 6,000 training sequences, evaluates on held-out students, compares against the app's previous parameter settings, and updates the app parameter artifact only if holdout calibration improves.

In [1]:
from __future__ import annotations

import csv
import json
import random
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

# Find the project root whether Jupyter starts here, in ml_research, or in notebooks.
ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "server" / "bkt-parameters.json").is_file()),
    Path.cwd(),
)
ML_DIR = ROOT / "ml_research"
sys.path.insert(0, str(ML_DIR))
from src.bkt import BKTModel

DATA_PATH = ML_DIR / "data" / "raw" / "skill_builder_data_corrected_collapsed.csv"
RESULTS_PATH = ML_DIR / "results" / "assistments_bkt.json"
PARAMETERS_PATH = ROOT / "server" / "bkt-parameters.json"
SEED = 42
TEST_FRACTION = 0.20
MAX_TRAIN_SEQUENCES = 6_000
MAX_ITERATIONS = 30
BASELINE = {"p_l0": 0.30, "p_t": 0.15, "p_g": 0.25, "p_s": 0.10}
DATASET_NAME = "ASSISTments 2009-2010 Skill Builder, corrected/collapsed version"
DATASET_URL = "https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download the corrected/collapsed CSV "
        "from the official ASSISTments page and save it at this path."
    )
print(f"Project root: {ROOT}")
print(f"Dataset file found: {DATA_PATH.name}")

Project root: C:\Users\Harish\Documents\Codex\2026-10-27\fir\outputs\echopilot
Dataset file found: skill_builder_data_corrected_collapsed.csv


## 1. Load and group response sequences

BKT learns from the order of correct/incorrect answers. We group each learner's answers by skill and sort them by `order_id`, the dataset's chronological problem-log field. We keep only the fields required for this experiment; student identifiers stay in memory and are not displayed or saved in the result files.

In [2]:
grouped_rows = defaultdict(list)
observed_skills = set()
valid_interactions = 0

# The historical CSV includes non-UTF-8 bytes, so latin-1 safely reads the fields.
with DATA_PATH.open("r", encoding="latin-1", newline="") as source:
    reader = csv.DictReader(source)
    required_columns = {"user_id", "skill_name", "skill_id", "correct", "order_id"}
    missing_columns = required_columns.difference(reader.fieldnames or [])
    if missing_columns:
        raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")

    for row in reader:
        student_id = (row.get("user_id") or "").strip()
        # Corrected multi-skill records use one composite skill label.
        skill = (row.get("skill_name") or "").strip() or (row.get("skill_id") or "").strip()
        outcome = (row.get("correct") or "").strip()
        if not student_id or not skill or outcome not in {"0", "1"}:
            continue
        try:
            order_id = int(float(row.get("order_id") or 0))
        except ValueError:
            order_id = valid_interactions
        grouped_rows[(student_id, skill)].append((order_id, int(outcome)))
        observed_skills.add(skill)
        valid_interactions += 1

responses_by_student = defaultdict(list)
for (student_id, _skill), ordered_rows in grouped_rows.items():
    ordered_rows.sort(key=lambda item: item[0])
    responses_by_student[student_id].append([outcome for _order, outcome in ordered_rows])

print(f"Valid interactions: {valid_interactions:,}")
print(f"Students: {len(responses_by_student):,}")
print(f"Observed skill labels: {len(observed_skills):,}")
print(f"Student-skill sequences: {sum(map(len, responses_by_student.values())):,}")

Valid interactions: 346,860
Students: 4,217
Observed skill labels: 113
Student-skill sequences: 39,280


## 2. Split students before training

A student-level split is important: if the same learner appeared on both sides, the holdout would be less independent. Seed 42 makes the split repeatable.

In [3]:
student_ids = sorted(responses_by_student)
random.Random(SEED).shuffle(student_ids)
test_count = max(1, round(len(student_ids) * TEST_FRACTION))
test_student_ids = set(student_ids[:test_count])
train_student_ids = set(student_ids[test_count:])

train_sequences = [
    sequence
    for student_id in sorted(train_student_ids)
    for sequence in responses_by_student[student_id]
]
test_sequences = [
    sequence
    for student_id in sorted(test_student_ids)
    for sequence in responses_by_student[student_id]
]

# This assertion checks the split design without revealing any IDs.
assert train_student_ids.isdisjoint(test_student_ids)
print(f"Train students: {len(train_student_ids):,}")
print(f"Held-out students: {len(test_student_ids):,}")
print(f"Available training sequences: {len(train_sequences):,}")
print(f"Held-out sequences: {len(test_sequences):,}")

Train students: 3,374
Held-out students: 843
Available training sequences: 31,322
Held-out sequences: 7,958


## 3. Fit BKT parameters with Baum-Welch EM

BKT models a hidden state (topic not yet known / topic known) and observes answer outcomes. Baum-Welch is an Expectation-Maximization algorithm: it estimates the likely hidden states, then updates the model's initial knowledge, learning, guess, and slip probabilities. It repeats those steps to improve the fit.

We cap training at a deterministic sample of 6,000 student-skill sequences to keep the experiment practical. The test sequences remain untouched during fitting.

In [4]:
training_sample = random.Random(SEED).sample(
    train_sequences, min(MAX_TRAIN_SEQUENCES, len(train_sequences))
)

fitted_model = BKTModel(
    p_l0=BASELINE["p_l0"],
    p_t=BASELINE["p_t"],
    p_g=BASELINE["p_g"],
    p_s=BASELINE["p_s"],
)
fitted_parameters = fitted_model.fit_em(training_sample, max_iter=MAX_ITERATIONS)
print(f"Training sequences used: {len(training_sample):,}")
print("Fitted parameters:", fitted_parameters)

Training sequences used: 6,000
Fitted parameters: {'p_l0': 0.7306, 'p_t': 0.0311, 'p_g': 0.3193, 'p_s': 0.2313}


## 4. Evaluate on held-out student responses

For each sequence, the model predicts the probability of the next observed answer before updating its mastery estimate with that answer. We compare the old app settings with the fitted BKT values on the same held-out responses.

In [5]:
def predict_responses(model, sequences):
    actual = []
    predicted = []
    for sequence in sequences:
        current_mastery = model.p_l0
        for outcome in sequence:
            predicted.append(model.predict_correct(current_mastery))
            actual.append(outcome)
            current_mastery = model.update_mastery(current_mastery, bool(outcome))
    return np.asarray(actual), np.clip(np.asarray(predicted), 1e-7, 1 - 1e-7)


def calculate_metrics(model, sequences):
    actual, predicted = predict_responses(model, sequences)
    scores = {
        "brier_score": round(float(brier_score_loss(actual, predicted)), 6),
        "log_loss": round(float(log_loss(actual, predicted, labels=[0, 1])), 6),
        "n_observations": int(len(actual)),
        "auc_roc": round(float(roc_auc_score(actual, predicted)), 6),
    }

    # ECE: weighted confidence/accuracy gap in ten probability bins.
    ece = 0.0
    for lower in np.linspace(0, 1, 11)[:-1]:
        upper = lower + 0.1
        in_bin = (predicted >= lower) & (
            predicted < upper if upper < 1 else predicted <= upper
        )
        if in_bin.any():
            ece += float(in_bin.mean()) * abs(
                float(actual[in_bin].mean() - predicted[in_bin].mean())
            )
    scores["ece"] = round(ece, 6)
    return scores


baseline_model = BKTModel(**BASELINE)
baseline_metrics = calculate_metrics(baseline_model, test_sequences)
fitted_metrics = calculate_metrics(fitted_model, test_sequences)

print(f"Held-out responses: {fitted_metrics['n_observations']:,}")
print("Former app settings:", baseline_metrics)
print("Fitted BKT:", fitted_metrics)

Held-out responses: 72,252
Former app settings: {'brier_score': 0.221969, 'log_loss': 0.653067, 'n_observations': 72252, 'auc_roc': 0.656065, 'ece': 0.104846}
Fitted BKT: {'brier_score': 0.206817, 'log_loss': 0.603175, 'n_observations': 72252, 'auc_roc': 0.659295, 'ece': 0.016676}


In [6]:
metric_labels = {
    "brier_score": "Brier score (lower is better)",
    "log_loss": "Log loss (lower is better)",
    "auc_roc": "AUC-ROC (higher is better)",
    "ece": "Expected calibration error (lower is better)",
}
print("| Metric | Former settings | Fitted BKT |")
print("|---|---:|---:|")
for key, label in metric_labels.items():
    print(f"| {label} | {baseline_metrics[key]:.6f} | {fitted_metrics[key]:.6f} |")

| Metric | Former settings | Fitted BKT |
|---|---:|---:|
| Brier score (lower is better) | 0.221969 | 0.206817 |
| Log loss (lower is better) | 0.653067 | 0.603175 |
| AUC-ROC (higher is better) | 0.656065 | 0.659295 |
| Expected calibration error (lower is better) | 0.104846 | 0.016676 |


## 5. Save aggregate results and app parameters

The notebook stores aggregate counts and metrics only. The raw interaction rows and student IDs are not exported. The app parameter artifact is updated only if the holdout Brier score is lower and log loss is no worse than the former settings.

In [7]:
use_fitted_parameters = (
    fitted_metrics["brier_score"] < baseline_metrics["brier_score"]
    and fitted_metrics["log_loss"] <= baseline_metrics["log_loss"]
)
created_at = datetime.now(timezone.utc).isoformat()
experiment = {
    "dataset": {
        "name": DATASET_NAME,
        "source": DATASET_URL,
        "valid_interactions": valid_interactions,
        "students": len(responses_by_student),
        "skills": len(observed_skills),
        "split": "deterministic student-level holdout",
        "seed": SEED,
        "test_fraction": TEST_FRACTION,
        "train_students": len(train_student_ids),
        "test_students": len(test_student_ids),
        "train_sequences_available": len(train_sequences),
        "train_sequences_used": len(training_sample),
        "test_sequences": len(test_sequences),
    },
    "baseline_parameters": BASELINE,
    "fitted_parameters": fitted_parameters,
    "held_out_metrics": {"baseline": baseline_metrics, "fitted": fitted_metrics},
    "integration_decision": "use_fitted_parameters" if use_fitted_parameters else "keep_app_defaults",
    "evaluation_note": "Lower Brier score and log loss indicate better probability forecasts; AUC and ECE are supplementary.",
    "created_at_utc": created_at,
}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(experiment, indent=2) + "\n", encoding="utf-8")
if use_fitted_parameters:
    app_artifact = {
        "parameters": {
            "initialKnowledge": fitted_parameters["p_l0"],
            "learning": fitted_parameters["p_t"],
            "guess": fitted_parameters["p_g"],
            "slip": fitted_parameters["p_s"],
        },
        "source": DATASET_NAME,
        "source_url": DATASET_URL,
        "trained_at_utc": created_at,
        "evaluation": experiment["held_out_metrics"],
    }
    PARAMETERS_PATH.write_text(json.dumps(app_artifact, indent=2) + "\n", encoding="utf-8")

print(f"Saved aggregate evaluation: {RESULTS_PATH.relative_to(ROOT)}")
print(f"Integration decision: {experiment['integration_decision']}")
if use_fitted_parameters:
    print(f"Updated runtime parameters: {PARAMETERS_PATH.relative_to(ROOT)}")

Saved aggregate evaluation: ml_research\results\assistments_bkt.json
Integration decision: use_fitted_parameters
Updated runtime parameters: server\bkt-parameters.json


## Interpretation and limitations

All four metrics improved on this student-level holdout, especially calibration (ECE). This is evidence that the fitted BKT probabilities better predicted answers in this historical math dataset. It does **not** show that EchoPilot improves learning, that Gemini questions are correct, or that the same values fit other subjects. The values are a shared starting prior; EchoPilot still updates each learner's progress from their own quiz answers.